In [55]:
import pandas as pd
import numpy as np
import pymssql
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pandas import json_normalize
# import requests

import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 1000) # Optional, to prevent wrapping
# pd.set_option('display.float_format', lambda x: '%.3f' % x)

# %% [markdown]
# **SQL Queries:**

# %%
# Establish parameters for connection: 
conn = pymssql.connect(
    server='SQL2022CL02',
    database='JDE_PRODUCTION'
)  

In [56]:
Transaction_SQL_Query = """
SELECT  [RPDOC] as "Document Number"
      ,(CAST([RPAAP] as float) / 100)  as "Combined Open Amount On As Of Date"
      ,[RPDCT] as "Document Type"
      ,[RPKCO] as "Document Company"
      ,DATEADD(DAY, ([RPDIVJ] % 1000) - 1, 
              DATEFROMPARTS( ([RPDIVJ] / 1000) + 1900, 1, 1)) as "Invoice Date"
      , (CAST([RPAG] as float) / 100) as "Gross Amount"
      ,Trim([RPMCU]) as "Business Unit"
      , CONCAT(LTRIM(STR([RPAN8], 20, 0)), ' - ', [RPALPH]) AS "Customer Number"
      ,DATEADD(DAY, ([RPJCL] % 1000) - 1, 
            DATEFROMPARTS( ([RPJCL] / 1000) + 1900, 1, 1)) as "Date Closed"   
      ,DATEADD(DAY, ([RPDDJ] % 1000) - 1, 
            DATEFROMPARTS( ([RPDDJ] / 1000) + 1900, 1, 1)) as "Due Date"   
      ,DATEADD(DAY, ([RPUPMJ] % 1000) - 1, 
      DATEFROMPARTS( ([RPUPMJ] / 1000) + 1900, 1, 1)) as "Date Last Updated"  
      ,[RPRMK] AS 'Remark'
      ,[RPVR01] AS 'Recording Comment'
      ,[RPRMR1] as 'Invoice Point of Contact'
      ,STR([RPICU]) AS 'Batch Number'
      ,[RPICUT] AS 'Batch Type'
 
      ,[RPDDNJ] AS 'Discount Due Date'
      ,[RPRDDJ] AS 'Date of Last Reminder'
      ,[RPPPDI] AS 'Invoice Print Date'
      ,[RPDTPB] AS 'Notification Payment Date'
      ,[RPHDGJ] AS 'Historical Date'
      ,[RPVDGJ] AS 'Void Date'
      ,[RPDGJ] AS 'GL Date'
      ,[RPDICJ] AS 'Batch Date'
      ,[RPDSVJ] AS 'Service Tax Date'
      ,[RPAID] AS 'Account ID'
      ,[RPDNLT] AS 'Delinquency Notice Sent'
      ,[RPRMDS] AS 'Number of Reminders Sent'
      ,[RPVOD] AS 'Void Flag'
      ,[RPRMR1] AS 'Invoice Point of Contact'

  FROM [JDE_PRODUCTION].[PRODDTA].[F03B11] 

  WHERE UPPER(LTRIM(RTRIM([F03B11].[RPMCU]))) NOT IN ('DS050', 'FIN14')
  AND (CAST([RPAG] as float) / 100) !=0
  And [RPDCT] != 'R5'
--  AND (CAST([RPAAP] as float) / 100) !=0


"""

# Load the queries: 

df_transaction = pd.read_sql_query(Transaction_SQL_Query,conn)

#Close the connection:

conn.close()

In [57]:
df_transaction

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact
0,1923.0,0.00,RM,06370,2019-09-11,-2900.00,ITD50,312721 - CITY OF NEWCASTLE,2019-09-23,2019-09-11,2019-09-23,2019 eCityGov Annual Mbrshp,Credit Memo,...,364079,IB,119254.0,0.0,0.0,0.0,0.0,0.0,119254.0,119254.0,119254.0,00007001,N,0.0,,...
1,1926.0,0.00,RM,01641,2019-10-07,-287.96,FIR50,249709 - WA ST Emergency Management Division,2019-10-23,2019-10-07,2019-10-23,249709 - WA ST EMERGENCY MANAG,Credit Memo for INV#35107,Krystal Hackmeister ...,366437,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...
2,1927.0,0.00,RM,01641,2019-10-07,-1965.85,FIR50,249709 - WA ST Emergency Management Division,2019-10-09,2019-10-07,2019-10-09,2016 UASI Reimbursement,Credit Memo INV#34389,Krystal Hackmeister ...,366438,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...
3,1938.0,0.00,RM,00100,2019-11-14,-709.20,PAR50,632354 - Kelly Edwards / LocalHOOPS Training Acad,2020-02-06,2019-11-14,2020-02-06,Commission for Sports camps,credit memo,...,368766,IB,119318.0,0.0,0.0,0.0,0.0,0.0,119318.0,119318.0,119318.0,00006956,Y,0.0,,...
4,1941.0,0.00,RM,01230,2019-12-31,-132774.52,UTL50,319933 - Republic Services,2020-02-05,2019-12-31,2020-02-05,CM for Sales Order30447,,Stephanie Schwenger ...,371356,IB,119365.0,0.0,0.0,0.0,0.0,0.0,119365.0,119365.0,119365.0,00006966,Y,0.0,,Stephanie Schwenger ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24674,96855.0,-0.20,RU,08050,2026-01-20,-15622.20,100,846036 - The Compliance Engine (TCE),1899-12-31,2026-01-20,2026-01-23,The Compliance Engine (TCE),THE COMPLIANCE ENGINE (TC,...,599612,RB,126020.0,0.0,0.0,0.0,126020.0,0.0,126020.0,126021.0,126020.0,00007028,Y,0.0,,...
24675,96874.0,0.00,RU,08050,2026-01-06,-1036759.45,100,338190 - WA ST DEPT OF COMMERCE,2026-01-06,2026-01-06,2026-01-23,Invoice 53091,,...,599984,RB,126006.0,0.0,0.0,0.0,126006.0,126006.0,126006.0,126023.0,126006.0,00007028,N,0.0,V,...
24676,96903.0,-1311.04,RU,08050,2026-01-26,-21337.64,100,41341 - KEMPER DEVELOPMENT,1899-12-31,2026-01-26,2026-01-27,KEMPER DEVELOPMENT,KEMPER DEVELOPMENT,...,600316,RB,126026.0,0.0,0.0,0.0,126026.0,0.0,126026.0,126027.0,126026.0,00007028,Y,0.0,,...
24677,97011.0,-200.00,RU,08050,2026-02-17,-200.00,100,41671 - Walgreen's # 03662,1899-12-31,2026-02-17,2026-02-18,Walgreen's # 03662,Walgreen's # 03662,...,602884,RB,126048.0,0.0,0.0,0.0,126048.0,0.0,126048.0,126049.0,126048.0,00007028,Y,0.0,,...


In [58]:
df_transaction[df_transaction["Document Number"]==50527]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact
18222,50527.0,324.0,RI,00100,2024-10-01,324.0,FIR01,1007888 - PUBLIC STORAGE - 1111 ...,1899-12-31,2024-10-31,2024-10-02,Fire Inspection Fee,,Fire Prevention @ 425-452-6872 ...,550879,IB,124305.0,125167.0,124276.0,125181.0,0.0,0.0,124275.0,124275.0,124275.0,00006956,Y,3.0,,Fire Prevention @ 425-452-6872 ...


In [59]:
# date column to date
date_cols = [
    "Invoice Date",
    "Due Date",
    "Date Closed"
]

df_transaction[date_cols] = df_transaction[date_cols].apply(lambda c: pd.to_datetime(c))

df_transaction["Document Number"] = df_transaction["Document Number"].astype("Int64").astype(str) 



In [60]:
df_transaction[df_transaction["Document Number"]== "50637"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact
18348,50637,0.0,RI,00100,2024-10-28,150.0,FIRSP,905488 - Hyatt Regency,2024-11-01,2024-11-27,2024-11-06,2nd False Alarms,in a 90 day time period,Fire Prevention @ 425-452-6872 ...,553386,IB,124332.0,0.0,125132.0,0.0,0.0,0.0,124302.0,124302.0,124302.0,00006956,Y,0.0,,Fire Prevention @ 425-452-6872 ...
18349,50637,200.0,RI,00100,2024-10-28,200.0,FIRSP,905488 - Hyatt Regency,1899-12-31,2024-11-27,2024-10-29,3rd False Alarms,in a 90 day time period,...,553386,IB,124332.0,125167.0,125132.0,125181.0,0.0,0.0,124302.0,124302.0,124302.0,00006956,Y,3.0,,...


In [ ]:
# test when all open invoice has open amount =0 
df_transaction[
    (df_transaction['Date Closed'] == pd.Timestamp('1899-12-31')) &
    (df_transaction['Combined Open Amount On As Of Date'] == 0)
]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status


In [61]:
# Categorize the invoices by Open, Closed: 

df_transaction['Status'] = np.where(df_transaction['Date Closed'] == pd.Timestamp('1899-12-31'), 'Open', 'Closed')

In [44]:
df_transaction

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status
0,1923,0.00,RM,06370,2019-09-11,-2900.00,ITD50,312721 - CITY OF NEWCASTLE,2019-09-23,2019-09-11,2019-09-23,2019 eCityGov Annual Mbrshp,Credit Memo,...,364079,IB,119254.0,0.0,0.0,0.0,0.0,0.0,119254.0,119254.0,119254.0,00007001,N,0.0,,...,Closed
1,1926,0.00,RM,01641,2019-10-07,-287.96,FIR50,249709 - WA ST Emergency Management Division,2019-10-23,2019-10-07,2019-10-23,249709 - WA ST EMERGENCY MANAG,Credit Memo for INV#35107,Krystal Hackmeister ...,366437,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed
2,1927,0.00,RM,01641,2019-10-07,-1965.85,FIR50,249709 - WA ST Emergency Management Division,2019-10-09,2019-10-07,2019-10-09,2016 UASI Reimbursement,Credit Memo INV#34389,Krystal Hackmeister ...,366438,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed
3,1938,0.00,RM,00100,2019-11-14,-709.20,PAR50,632354 - Kelly Edwards / LocalHOOPS Training Acad,2020-02-06,2019-11-14,2020-02-06,Commission for Sports camps,credit memo,...,368766,IB,119318.0,0.0,0.0,0.0,0.0,0.0,119318.0,119318.0,119318.0,00006956,Y,0.0,,...,Closed
4,1941,0.00,RM,01230,2019-12-31,-132774.52,UTL50,319933 - Republic Services,2020-02-05,2019-12-31,2020-02-05,CM for Sales Order30447,,Stephanie Schwenger ...,371356,IB,119365.0,0.0,0.0,0.0,0.0,0.0,119365.0,119365.0,119365.0,00006966,Y,0.0,,Stephanie Schwenger ...,Closed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24668,96855,-0.20,RU,08050,2026-01-20,-15622.20,100,846036 - The Compliance Engine (TCE),1899-12-31,2026-01-20,2026-01-23,The Compliance Engine (TCE),THE COMPLIANCE ENGINE (TC,...,599612,RB,126020.0,0.0,0.0,0.0,126020.0,0.0,126020.0,126021.0,126020.0,00007028,Y,0.0,,...,Open
24669,96874,0.00,RU,08050,2026-01-06,-1036759.45,100,338190 - WA ST DEPT OF COMMERCE,2026-01-06,2026-01-06,2026-01-23,Invoice 53091,,...,599984,RB,126006.0,0.0,0.0,0.0,126006.0,126006.0,126006.0,126023.0,126006.0,00007028,N,0.0,V,...,Closed
24670,96903,-1311.04,RU,08050,2026-01-26,-21337.64,100,41341 - KEMPER DEVELOPMENT,1899-12-31,2026-01-26,2026-01-27,KEMPER DEVELOPMENT,KEMPER DEVELOPMENT,...,600316,RB,126026.0,0.0,0.0,0.0,126026.0,0.0,126026.0,126027.0,126026.0,00007028,Y,0.0,,...,Open
24671,97011,-200.00,RU,08050,2026-02-17,-200.00,100,41671 - Walgreen's # 03662,1899-12-31,2026-02-17,2026-02-18,Walgreen's # 03662,Walgreen's # 03662,...,602884,RB,126048.0,0.0,0.0,0.0,126048.0,0.0,126048.0,126049.0,126048.0,00007028,Y,0.0,,...,Open


### BusinessUnit mapping table

In [45]:
df_busi_unit_desc = pd.DataFrame(
    [
        ("FIR50", "Fire"),
        ("TRA50", "Transportation"),
        ("CDD50", "Community Development"),
        ("100",   "AR-Research"),
        ("FIR01", "Fire Prev"),
        ("FIROP", "Fire Prev"),
        ("FIRSP", "Fire Prev"),
        ("FIN50", "Tax"),
        ("PAR50", "Parks"),
        ("ITD50", "IT"),
        ("PARAQ", "Aquatics"),
        ("FIR04", "Fire Prev"),
        ("POL50", "Police"),
        ("FAM50", "FAM"),
        ("DSD50", "Development Services"),
        ("UTL50", "Utilities"),
        ("ARC50", "ARCH"),
        ("CMO50", "City Manager Office"),
        ("HR050", "HR"),
        # ("CAO50", "City Attorneys Office"),   
        # ("PARMA", "Marina"),
        # ("HR001", "HR"),
        # ("HR002", "HR"),
    ],
    columns=["Code", "BusinessUnitDesc"]
)

# Normalize the code key (trim + uppercase)
df_busi_unit_desc["Code"] = df_busi_unit_desc["Code"].astype(str)
df_busi_unit_desc

,Code,BusinessUnitDesc
0,FIR50,Fire
1,TRA50,Transportation
2,CDD50,Community Development
3,100,AR-Research
4,FIR01,Fire Prev
5,FIROP,Fire Prev
6,FIRSP,Fire Prev
7,FIN50,Tax
8,PAR50,Parks
9,ITD50,IT


In [46]:
# Left join to bring in description
df_transaction = df_transaction.merge(
    df_busi_unit_desc[["Code", "BusinessUnitDesc"]],
    how="left",
    left_on="Business Unit",
    right_on="Code"
)



In [47]:
# drop key columns
df_transaction.drop(columns=["Code"], inplace=True)

In [48]:
df_transaction

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc
0,1923,0.00,RM,06370,2019-09-11,-2900.00,ITD50,312721 - CITY OF NEWCASTLE,2019-09-23,2019-09-11,2019-09-23,2019 eCityGov Annual Mbrshp,Credit Memo,...,364079,IB,119254.0,0.0,0.0,0.0,0.0,0.0,119254.0,119254.0,119254.0,00007001,N,0.0,,...,Closed,IT
1,1926,0.00,RM,01641,2019-10-07,-287.96,FIR50,249709 - WA ST Emergency Management Division,2019-10-23,2019-10-07,2019-10-23,249709 - WA ST EMERGENCY MANAG,Credit Memo for INV#35107,Krystal Hackmeister ...,366437,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed,Fire
2,1927,0.00,RM,01641,2019-10-07,-1965.85,FIR50,249709 - WA ST Emergency Management Division,2019-10-09,2019-10-07,2019-10-09,2016 UASI Reimbursement,Credit Memo INV#34389,Krystal Hackmeister ...,366438,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed,Fire
3,1938,0.00,RM,00100,2019-11-14,-709.20,PAR50,632354 - Kelly Edwards / LocalHOOPS Training Acad,2020-02-06,2019-11-14,2020-02-06,Commission for Sports camps,credit memo,...,368766,IB,119318.0,0.0,0.0,0.0,0.0,0.0,119318.0,119318.0,119318.0,00006956,Y,0.0,,...,Closed,Parks
4,1941,0.00,RM,01230,2019-12-31,-132774.52,UTL50,319933 - Republic Services,2020-02-05,2019-12-31,2020-02-05,CM for Sales Order30447,,Stephanie Schwenger ...,371356,IB,119365.0,0.0,0.0,0.0,0.0,0.0,119365.0,119365.0,119365.0,00006966,Y,0.0,,Stephanie Schwenger ...,Closed,Utilities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24668,96855,-0.20,RU,08050,2026-01-20,-15622.20,100,846036 - The Compliance Engine (TCE),1899-12-31,2026-01-20,2026-01-23,The Compliance Engine (TCE),THE COMPLIANCE ENGINE (TC,...,599612,RB,126020.0,0.0,0.0,0.0,126020.0,0.0,126020.0,126021.0,126020.0,00007028,Y,0.0,,...,Open,AR-Research
24669,96874,0.00,RU,08050,2026-01-06,-1036759.45,100,338190 - WA ST DEPT OF COMMERCE,2026-01-06,2026-01-06,2026-01-23,Invoice 53091,,...,599984,RB,126006.0,0.0,0.0,0.0,126006.0,126006.0,126006.0,126023.0,126006.0,00007028,N,0.0,V,...,Closed,AR-Research
24670,96903,-1311.04,RU,08050,2026-01-26,-21337.64,100,41341 - KEMPER DEVELOPMENT,1899-12-31,2026-01-26,2026-01-27,KEMPER DEVELOPMENT,KEMPER DEVELOPMENT,...,600316,RB,126026.0,0.0,0.0,0.0,126026.0,0.0,126026.0,126027.0,126026.0,00007028,Y,0.0,,...,Open,AR-Research
24671,97011,-200.00,RU,08050,2026-02-17,-200.00,100,41671 - Walgreen's # 03662,1899-12-31,2026-02-17,2026-02-18,Walgreen's # 03662,Walgreen's # 03662,...,602884,RB,126048.0,0.0,0.0,0.0,126048.0,0.0,126048.0,126049.0,126048.0,00007028,Y,0.0,,...,Open,AR-Research


### 1

In [49]:
# Mapping Document Type(RI: invoice; RU: Overpay；RM:Credit)
doc_type_map = {
    "RI": "Invoice",
    "RU": "Overpay",
    "RM": "Credit"
}

df_transaction["Document Type"] = df_transaction["Document Type"].map(doc_type_map).fillna(df_transaction["Document Type"])

In [50]:
# Create "Days Past Due" column, today- Due date
today = pd.Timestamp.today().normalize()

df_transaction["Days Past Due"] = (today - df_transaction["Due Date"]).dt.days

In [51]:
df_transaction

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due
0,1923,0.00,Credit,06370,2019-09-11,-2900.00,ITD50,312721 - CITY OF NEWCASTLE,2019-09-23,2019-09-11,2019-09-23,2019 eCityGov Annual Mbrshp,Credit Memo,...,364079,IB,119254.0,0.0,0.0,0.0,0.0,0.0,119254.0,119254.0,119254.0,00007001,N,0.0,,...,Closed,IT,2366
1,1926,0.00,Credit,01641,2019-10-07,-287.96,FIR50,249709 - WA ST Emergency Management Division,2019-10-23,2019-10-07,2019-10-23,249709 - WA ST EMERGENCY MANAG,Credit Memo for INV#35107,Krystal Hackmeister ...,366437,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed,Fire,2340
2,1927,0.00,Credit,01641,2019-10-07,-1965.85,FIR50,249709 - WA ST Emergency Management Division,2019-10-09,2019-10-07,2019-10-09,2016 UASI Reimbursement,Credit Memo INV#34389,Krystal Hackmeister ...,366438,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed,Fire,2340
3,1938,0.00,Credit,00100,2019-11-14,-709.20,PAR50,632354 - Kelly Edwards / LocalHOOPS Training Acad,2020-02-06,2019-11-14,2020-02-06,Commission for Sports camps,credit memo,...,368766,IB,119318.0,0.0,0.0,0.0,0.0,0.0,119318.0,119318.0,119318.0,00006956,Y,0.0,,...,Closed,Parks,2302
4,1941,0.00,Credit,01230,2019-12-31,-132774.52,UTL50,319933 - Republic Services,2020-02-05,2019-12-31,2020-02-05,CM for Sales Order30447,,Stephanie Schwenger ...,371356,IB,119365.0,0.0,0.0,0.0,0.0,0.0,119365.0,119365.0,119365.0,00006966,Y,0.0,,Stephanie Schwenger ...,Closed,Utilities,2255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24668,96855,-0.20,Overpay,08050,2026-01-20,-15622.20,100,846036 - The Compliance Engine (TCE),1899-12-31,2026-01-20,2026-01-23,The Compliance Engine (TCE),THE COMPLIANCE ENGINE (TC,...,599612,RB,126020.0,0.0,0.0,0.0,126020.0,0.0,126020.0,126021.0,126020.0,00007028,Y,0.0,,...,Open,AR-Research,43
24669,96874,0.00,Overpay,08050,2026-01-06,-1036759.45,100,338190 - WA ST DEPT OF COMMERCE,2026-01-06,2026-01-06,2026-01-23,Invoice 53091,,...,599984,RB,126006.0,0.0,0.0,0.0,126006.0,126006.0,126006.0,126023.0,126006.0,00007028,N,0.0,V,...,Closed,AR-Research,57
24670,96903,-1311.04,Overpay,08050,2026-01-26,-21337.64,100,41341 - KEMPER DEVELOPMENT,1899-12-31,2026-01-26,2026-01-27,KEMPER DEVELOPMENT,KEMPER DEVELOPMENT,...,600316,RB,126026.0,0.0,0.0,0.0,126026.0,0.0,126026.0,126027.0,126026.0,00007028,Y,0.0,,...,Open,AR-Research,37
24671,97011,-200.00,Overpay,08050,2026-02-17,-200.00,100,41671 - Walgreen's # 03662,1899-12-31,2026-02-17,2026-02-18,Walgreen's # 03662,Walgreen's # 03662,...,602884,RB,126048.0,0.0,0.0,0.0,126048.0,0.0,126048.0,126049.0,126048.0,00007028,Y,0.0,,...,Open,AR-Research,15


In [52]:
df_transaction[df_transaction["Document Number"]=="48823"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due
16039,48823,0.00,Invoice,00100,2024-03-28,290.09,TRA50,960517 - Verizon Wireless,2024-07-03,2024-04-27,2024-07-04,Prorated Pole Lease Fee,23-107589 TJ,Cassie Davis 425.452.4550 ...,532551,IB,124118.0,124155.0,124353.0,124182.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,1.0,,Cassie Davis 425.452.4550 ...,Closed,Transportation,676
16040,48823,385.44,Invoice,00100,2024-03-28,385.52,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-07-04,Prorated Est. Pwr Usage Fee,23-107589 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676
16041,48823,454.20,Invoice,00100,2024-03-28,454.20,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-06-07,2024 Est. Pwr Usage Fee,21-134141 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676
16042,48823,427.70,Invoice,00100,2024-03-28,427.70,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-06-07,2024 Est. Pwr Usage Fee,22-118579 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676
16043,48823,427.70,Invoice,00100,2024-03-28,427.70,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-06-07,2024 Est. Pwr Usage Fee,22-127966 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676
16044,48823,1100.40,Invoice,00100,2024-03-28,1100.40,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-06-07,2024 Power Access Fee,22-127966 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676


In [53]:
df_transaction[df_transaction["Document Number"]=="43394"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due
9441,43394,185000.0,Invoice,03680,2022-09-07,1850000.0,PAR50,412661 - WA ST RECREATION AND CONSERVATION OFFICE,1899-12-31,2022-10-07,2023-07-07,RCO Lake Samm Grant Reimb.,,Camron Parker - 425-452-2032 ...,481139,IB,122280.0,0.0,123011.0,0.0,0.0,0.0,122250.0,122250.0,122250.0,00006979,N,0.0,,Camron Parker - 425-452-2032 ...,Open,Parks,1244


In [54]:
df_transaction[df_transaction["Document Number"]=="48823"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due
16039,48823,0.00,Invoice,00100,2024-03-28,290.09,TRA50,960517 - Verizon Wireless,2024-07-03,2024-04-27,2024-07-04,Prorated Pole Lease Fee,23-107589 TJ,Cassie Davis 425.452.4550 ...,532551,IB,124118.0,124155.0,124353.0,124182.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,1.0,,Cassie Davis 425.452.4550 ...,Closed,Transportation,676
16040,48823,385.44,Invoice,00100,2024-03-28,385.52,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-07-04,Prorated Est. Pwr Usage Fee,23-107589 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676
16041,48823,454.20,Invoice,00100,2024-03-28,454.20,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-06-07,2024 Est. Pwr Usage Fee,21-134141 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676
16042,48823,427.70,Invoice,00100,2024-03-28,427.70,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-06-07,2024 Est. Pwr Usage Fee,22-118579 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676
16043,48823,427.70,Invoice,00100,2024-03-28,427.70,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-06-07,2024 Est. Pwr Usage Fee,22-127966 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676
16044,48823,1100.40,Invoice,00100,2024-03-28,1100.40,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-06-07,2024 Power Access Fee,22-127966 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676


### Group by document number to get total due amount

In [17]:
# Create "Total Open Amount per Document" column: group by those open invoices and sum the outstanding amount, closed invoices show 0

open_total = (
    df_transaction["Gross Amount"]
    .where(df_transaction["Status"] == "Open", 0)
    .groupby(df_transaction["Document Number"])
    .transform("sum")
)

df_transaction["Total Open Amount per Document"] = open_total.where(
    df_transaction["Status"] == "Open", 0
)
df_transaction

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due,Total Open Amount per Document
0,1923,0.00,Credit,06370,2019-09-11,-2900.00,ITD50,312721 - CITY OF NEWCASTLE,2019-09-23,2019-09-11,2019-09-23,2019 eCityGov Annual Mbrshp,Credit Memo,...,364079,IB,119254.0,0.0,0.0,0.0,0.0,0.0,119254.0,119254.0,119254.0,00007001,N,0.0,,...,Closed,IT,2366,0.00
1,1926,0.00,Credit,01641,2019-10-07,-287.96,FIR50,249709 - WA ST Emergency Management Division,2019-10-23,2019-10-07,2019-10-23,249709 - WA ST EMERGENCY MANAG,Credit Memo for INV#35107,Krystal Hackmeister ...,366437,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed,Fire,2340,0.00
2,1927,0.00,Credit,01641,2019-10-07,-1965.85,FIR50,249709 - WA ST Emergency Management Division,2019-10-09,2019-10-07,2019-10-09,2016 UASI Reimbursement,Credit Memo INV#34389,Krystal Hackmeister ...,366438,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed,Fire,2340,0.00
3,1938,0.00,Credit,00100,2019-11-14,-709.20,PAR50,632354 - Kelly Edwards / LocalHOOPS Training Acad,2020-02-06,2019-11-14,2020-02-06,Commission for Sports camps,credit memo,...,368766,IB,119318.0,0.0,0.0,0.0,0.0,0.0,119318.0,119318.0,119318.0,00006956,Y,0.0,,...,Closed,Parks,2302,0.00
4,1941,0.00,Credit,01230,2019-12-31,-132774.52,UTL50,319933 - Republic Services,2020-02-05,2019-12-31,2020-02-05,CM for Sales Order30447,,Stephanie Schwenger ...,371356,IB,119365.0,0.0,0.0,0.0,0.0,0.0,119365.0,119365.0,119365.0,00006966,Y,0.0,,Stephanie Schwenger ...,Closed,Utilities,2255,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24668,96855,-0.20,Overpay,08050,2026-01-20,-15622.20,100,846036 - The Compliance Engine (TCE),1899-12-31,2026-01-20,2026-01-23,The Compliance Engine (TCE),THE COMPLIANCE ENGINE (TC,...,599612,RB,126020.0,0.0,0.0,0.0,126020.0,0.0,126020.0,126021.0,126020.0,00007028,Y,0.0,,...,Open,AR-Research,43,-15622.20
24669,96874,0.00,Overpay,08050,2026-01-06,-1036759.45,100,338190 - WA ST DEPT OF COMMERCE,2026-01-06,2026-01-06,2026-01-23,Invoice 53091,,...,599984,RB,126006.0,0.0,0.0,0.0,126006.0,126006.0,126006.0,126023.0,126006.0,00007028,N,0.0,V,...,Closed,AR-Research,57,0.00
24670,96903,-1311.04,Overpay,08050,2026-01-26,-21337.64,100,41341 - KEMPER DEVELOPMENT,1899-12-31,2026-01-26,2026-01-27,KEMPER DEVELOPMENT,KEMPER DEVELOPMENT,...,600316,RB,126026.0,0.0,0.0,0.0,126026.0,0.0,126026.0,126027.0,126026.0,00007028,Y,0.0,,...,Open,AR-Research,37,-21337.64
24671,97011,-200.00,Overpay,08050,2026-02-17,-200.00,100,41671 - Walgreen's # 03662,1899-12-31,2026-02-17,2026-02-18,Walgreen's # 03662,Walgreen's # 03662,...,602884,RB,126048.0,0.0,0.0,0.0,126048.0,0.0,126048.0,126049.0,126048.0,00007028,Y,0.0,,...,Open,AR-Research,15,-200.00


In [18]:
# keep only one line for each open invoice 

df_closed = df_transaction[df_transaction["Status"] != "Open"]

df_open_single = (
    df_transaction[df_transaction["Status"] == "Open"]
    .sort_values("Document Number")
    .drop_duplicates(subset="Document Number", keep="first")
)

df_transaction = pd.concat([df_closed, df_open_single], ignore_index=True)

In [19]:
df_transaction

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due,Total Open Amount per Document
0,1923,0.00,Credit,06370,2019-09-11,-2900.00,ITD50,312721 - CITY OF NEWCASTLE,2019-09-23,2019-09-11,2019-09-23,2019 eCityGov Annual Mbrshp,Credit Memo,...,364079,IB,119254.0,0.0,0.0,0.0,0.0,0.0,119254.0,119254.0,119254.0,00007001,N,0.0,,...,Closed,IT,2366,0.00
1,1926,0.00,Credit,01641,2019-10-07,-287.96,FIR50,249709 - WA ST Emergency Management Division,2019-10-23,2019-10-07,2019-10-23,249709 - WA ST EMERGENCY MANAG,Credit Memo for INV#35107,Krystal Hackmeister ...,366437,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed,Fire,2340,0.00
2,1927,0.00,Credit,01641,2019-10-07,-1965.85,FIR50,249709 - WA ST Emergency Management Division,2019-10-09,2019-10-07,2019-10-09,2016 UASI Reimbursement,Credit Memo INV#34389,Krystal Hackmeister ...,366438,IB,119280.0,0.0,0.0,0.0,0.0,0.0,119280.0,119280.0,119280.0,03154071,N,0.0,,Krystal Hackmeister ...,Closed,Fire,2340,0.00
3,1938,0.00,Credit,00100,2019-11-14,-709.20,PAR50,632354 - Kelly Edwards / LocalHOOPS Training Acad,2020-02-06,2019-11-14,2020-02-06,Commission for Sports camps,credit memo,...,368766,IB,119318.0,0.0,0.0,0.0,0.0,0.0,119318.0,119318.0,119318.0,00006956,Y,0.0,,...,Closed,Parks,2302,0.00
4,1941,0.00,Credit,01230,2019-12-31,-132774.52,UTL50,319933 - Republic Services,2020-02-05,2019-12-31,2020-02-05,CM for Sales Order30447,,Stephanie Schwenger ...,371356,IB,119365.0,0.0,0.0,0.0,0.0,0.0,119365.0,119365.0,119365.0,00006966,Y,0.0,,Stephanie Schwenger ...,Closed,Utilities,2255,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24371,96678,-117.00,Overpay,08050,2025-12-12,-117.00,100,41973 - Chinook Aquatic Club,1899-12-31,2025-12-12,2025-12-15,Chinook Aquatic Club,Chinook Aquatic Club,...,595854,RB,125346.0,0.0,0.0,0.0,125346.0,0.0,125346.0,125349.0,125346.0,00007028,Y,0.0,,...,Open,AR-Research,82,-117.00
24372,96837,-610.63,Overpay,08050,2026-01-15,-610.63,100,309010 - Waterbabies,1899-12-31,2026-01-15,2026-01-16,Waterbabies,Waterbabies,...,599264,RB,126015.0,0.0,0.0,0.0,126015.0,0.0,126015.0,126016.0,126015.0,00007028,Y,0.0,,...,Open,AR-Research,48,-610.63
24373,96855,-0.20,Overpay,08050,2026-01-20,-15622.20,100,846036 - The Compliance Engine (TCE),1899-12-31,2026-01-20,2026-01-23,The Compliance Engine (TCE),THE COMPLIANCE ENGINE (TC,...,599612,RB,126020.0,0.0,0.0,0.0,126020.0,0.0,126020.0,126021.0,126020.0,00007028,Y,0.0,,...,Open,AR-Research,43,-15622.20
24374,96903,-1311.04,Overpay,08050,2026-01-26,-21337.64,100,41341 - KEMPER DEVELOPMENT,1899-12-31,2026-01-26,2026-01-27,KEMPER DEVELOPMENT,KEMPER DEVELOPMENT,...,600316,RB,126026.0,0.0,0.0,0.0,126026.0,0.0,126026.0,126027.0,126026.0,00007028,Y,0.0,,...,Open,AR-Research,37,-21337.64


In [20]:
df_transaction[df_transaction["Document Number"]=="49825"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due,Total Open Amount per Document
17087,49825,0.0,Invoice,04250,2024-07-10,277.12,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,Monthly Slip rent July 11-31 ',,Kim Bui - 425-452-4317 ...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,Kim Bui - 425-452-4317 ...,Closed,NaN,572,0.0
17088,49825,0.0,Invoice,04250,2024-07-10,35.58,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,LHET July 2024,,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0
17089,49825,0.0,Invoice,04250,2024-07-10,409.08,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,Monthly Slip Rental Aug 2024,,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0
17090,49825,0.0,Invoice,04250,2024-07-10,52.53,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,LHET August 2024,,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0
17091,49825,0.0,Invoice,04250,2024-07-10,409.08,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,Monthly Slip Rental Sept 2024,,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0
17092,49825,0.0,Invoice,04250,2024-07-10,52.53,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,LHET Sept 2024,,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0
17093,49825,0.0,Invoice,04250,2024-07-10,171.55,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,Monthly Slip Rent Oct 1-13 '24,,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0
17094,49825,0.0,Invoice,04250,2024-07-10,22.03,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,LHET Oct 2024,,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0
17095,49825,0.0,Invoice,04250,2024-07-10,500.00,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,Concession Permit,One Time Payment,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0
17096,49825,0.0,Invoice,04250,2024-07-10,187.65,PARMA,989905 - Duffy Boat - ACIRS LLC,2024-07-10,2024-08-09,2024-07-11,Whaling Rent July 11-31 '24,,...,542649,IB,124222.0,0.0,124191.0,0.0,0.0,0.0,124192.0,124191.0,124192.0,00556025,Y,0.0,,...,Closed,NaN,572,0.0


In [35]:
df_transaction[df_transaction["Document Number"]=="43394"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due,Total Open Amount per Document,Aging Bucket,IsClosedLine
22776,43394,185000.0,Invoice,03680,2022-09-07,1850000.0,PAR50,412661 - WA ST RECREATION AND CONSERVATION OFFICE,1899-12-31,2022-10-07,2023-07-07,RCO Lake Samm Grant Reimb.,,Camron Parker - 425-452-2032 ...,481139,IB,122280.0,0.0,123011.0,0.0,0.0,0.0,122250.0,122250.0,122250.0,00006979,N,0.0,,Camron Parker - 425-452-2032 ...,Open,Parks,1244,1850000.0,121+ Days,False


In [21]:
df_transaction[df_transaction["Document Number"]=="48823"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due,Total Open Amount per Document
15856,48823,0.00,Invoice,00100,2024-03-28,290.09,TRA50,960517 - Verizon Wireless,2024-07-03,2024-04-27,2024-07-04,Prorated Pole Lease Fee,23-107589 TJ,Cassie Davis 425.452.4550 ...,532551,IB,124118.0,124155.0,124353.0,124182.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,1.0,,Cassie Davis 425.452.4550 ...,Closed,Transportation,676,0.00
22904,48823,385.44,Invoice,00100,2024-03-28,385.52,TRA50,960517 - Verizon Wireless,1899-12-31,2024-04-27,2024-07-04,Prorated Est. Pwr Usage Fee,23-107589 TJ,...,532551,IB,124118.0,124225.0,124353.0,124244.0,0.0,0.0,124088.0,124088.0,124088.0,00006956,Y,3.0,,...,Open,Transportation,676,2795.52


In [22]:
# df_transaction =df_transaction.drop_duplicates(subset="Document Number").reset_index(drop=True)

In [23]:
# df_transaction.drop(columns="Gross Amount", inplace=True)


### Aging Bucket

In [24]:
bins = [-float("inf"), 0, 30, 60, 90, 120, float("inf")]
labels = [
    "Current",
    "1 to 30 Days",
    "31 to 60 Days",
    "61 to 90 Days",
    "91 to 120 Days",
    "121+ Days"
]

df_transaction["Aging Bucket"] = pd.cut(
    df_transaction["Days Past Due"],
    bins=bins,
    labels=labels
)

In [25]:
df_transaction[df_transaction["Status"]=="Open"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due,Total Open Amount per Document,Aging Bucket
22739,2200,-150.00,Credit,00100,2023-12-13,-150.00,FIRSP,70268 - St Lukes Lutheran Church,1899-12-31,2023-12-13,2023-12-14,Cancel Inv 47166,Previously Paid,Kyle Nelson ...,522095,IB,123347.0,0.0,0.0,0.0,0.0,0.0,123347.0,123347.0,123347.0,00006956,Y,0.0,,Kyle Nelson ...,Open,Fire Prev,812,-150.00,121+ Days
22740,2208,-342.00,Credit,01430,2023-12-31,-342.00,PARAQ,605640 - SEATTLE SYNCHRO,1899-12-31,2023-12-31,2024-02-06,Overbill Dec 2023,4Hrs @85.50/hr,Patrick Simmons 425.452.4102 ...,527156,IB,123365.0,0.0,124040.0,0.0,0.0,0.0,123365.0,124037.0,123365.0,00006972,Y,0.0,,Patrick Simmons 425.452.4102 ...,Open,Aquatics,794,-342.00,121+ Days
22741,2210,-400.00,Credit,00100,2024-02-15,-400.00,FIRSP,884780 - Avalon Towers Bellevue,1899-12-31,2024-02-15,2024-02-15,False Alarm reversal,wrong location,Fire Prevention @ 425-452-6872 ...,528113,IB,124046.0,0.0,124046.0,0.0,0.0,0.0,124046.0,124046.0,124046.0,00006956,Y,0.0,,Fire Prevention @ 425-452-6872 ...,Open,Fire Prev,748,-400.00,121+ Days
22742,2232,-276.16,Credit,00100,2024-12-18,-276.16,FAM50,257445 - CROWN CASTLE,1899-12-31,2024-12-18,2025-05-09,Orig amt billed in Error,,Andrea Jutte ...,558501,IB,124353.0,0.0,124353.0,0.0,0.0,0.0,124353.0,124353.0,124353.0,00006956,N,0.0,,Andrea Jutte ...,Open,FAM,441,-276.16,121+ Days
22743,2233,-276.16,Credit,00100,2024-12-18,-276.16,FAM50,257445 - CROWN CASTLE,1899-12-31,2024-12-18,2025-05-09,Amt billed in error,,Andrea Jutte ...,558501,IB,124353.0,0.0,124353.0,0.0,0.0,0.0,124353.0,124353.0,124353.0,00006956,N,0.0,,Andrea Jutte ...,Open,FAM,441,-276.16,121+ Days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24371,96678,-117.00,Overpay,08050,2025-12-12,-117.00,100,41973 - Chinook Aquatic Club,1899-12-31,2025-12-12,2025-12-15,Chinook Aquatic Club,Chinook Aquatic Club,...,595854,RB,125346.0,0.0,0.0,0.0,125346.0,0.0,125346.0,125349.0,125346.0,00007028,Y,0.0,,...,Open,AR-Research,82,-117.00,61 to 90 Days
24372,96837,-610.63,Overpay,08050,2026-01-15,-610.63,100,309010 - Waterbabies,1899-12-31,2026-01-15,2026-01-16,Waterbabies,Waterbabies,...,599264,RB,126015.0,0.0,0.0,0.0,126015.0,0.0,126015.0,126016.0,126015.0,00007028,Y,0.0,,...,Open,AR-Research,48,-610.63,31 to 60 Days
24373,96855,-0.20,Overpay,08050,2026-01-20,-15622.20,100,846036 - The Compliance Engine (TCE),1899-12-31,2026-01-20,2026-01-23,The Compliance Engine (TCE),THE COMPLIANCE ENGINE (TC,...,599612,RB,126020.0,0.0,0.0,0.0,126020.0,0.0,126020.0,126021.0,126020.0,00007028,Y,0.0,,...,Open,AR-Research,43,-15622.20,31 to 60 Days
24374,96903,-1311.04,Overpay,08050,2026-01-26,-21337.64,100,41341 - KEMPER DEVELOPMENT,1899-12-31,2026-01-26,2026-01-27,KEMPER DEVELOPMENT,KEMPER DEVELOPMENT,...,600316,RB,126026.0,0.0,0.0,0.0,126026.0,0.0,126026.0,126027.0,126026.0,00007028,Y,0.0,,...,Open,AR-Research,37,-21337.64,31 to 60 Days


In [26]:
df_transaction[df_transaction["Document Number"]=="50527"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due,Total Open Amount per Document,Aging Bucket
22979,50527,324.0,Invoice,00100,2024-10-01,324.0,FIR01,1007888 - PUBLIC STORAGE - 1111 ...,1899-12-31,2024-10-31,2024-10-02,Fire Inspection Fee,,Fire Prevention @ 425-452-6872 ...,550879,IB,124305.0,125167.0,124276.0,125181.0,0.0,0.0,124275.0,124275.0,124275.0,00006956,Y,3.0,,Fire Prevention @ 425-452-6872 ...,Open,Fire Prev,489,324.0,121+ Days


### Closed Invoices 

In [ ]:
# # Days to Pay using existing Status column (Open/Closed)

# - An invoice is paid in full only if ALL its line items are Closed
# - PaidDate = latest Date Closed among the lines
# - DaysToPay = PaidDate - Due Date (invoice-level)

df_transaction["IsClosedLine"] = (df_transaction["Status"] == "Closed")

df_ClosedInvoice = (
    df_transaction.groupby("Document Number", as_index=False)
    .agg(
        InvoiceDate=("Invoice Date", "first"),
        BusinessUnitDesc=("BusinessUnitDesc", "first"),
        CustomerNumber=("Customer Number", "first"),
        DueDate=("Due Date", "first"),
        PaidInFull=("IsClosedLine", "all"),   # all lines closed
        PaidDate=("Date Closed", "max"),      # last close date across lines
        LineCount=("Document Number", "size"),
        ClosedLineCount=("IsClosedLine", "sum"),
    )
)

# Days to pay: only compute when PaidInFull is True ---
df_ClosedInvoice["DaysToPay_FromDue"] = (df_ClosedInvoice["PaidDate"] - df_ClosedInvoice["DueDate"]).dt.days
#If the invoice is not paid in full, then replace DaysToPay_FromDue replace with NA
df_ClosedInvoice.loc[~df_ClosedInvoice["PaidInFull"], "DaysToPay_FromDue"] = pd.NA
df_ClosedInvoice


,Document Number,InvoiceDate,BusinessUnitDesc,CustomerNumber,DueDate,PaidInFull,PaidDate,LineCount,ClosedLineCount,DaysToPay_FromDue
0,1923,2019-09-11,IT,312721 - CITY OF NEWCASTLE,2019-09-11,True,2019-09-23,1,1,12.0
1,1926,2019-10-07,Fire,249709 - WA ST Emergency Management Division,2019-10-07,True,2019-10-23,1,1,16.0
2,1927,2019-10-07,Fire,249709 - WA ST Emergency Management Division,2019-10-07,True,2019-10-09,1,1,2.0
3,1938,2019-11-14,Parks,632354 - Kelly Edwards / LocalHOOPS Training Acad,2019-11-14,True,2020-02-06,1,1,84.0
4,1941,2019-12-31,Utilities,319933 - Republic Services,2019-12-31,True,2020-02-05,1,1,36.0
...,...,...,...,...,...,...,...,...,...,...
18912,96855,2026-01-20,AR-Research,846036 - The Compliance Engine (TCE),2026-01-20,False,1899-12-31,1,0,NaN
18913,96874,2026-01-06,AR-Research,338190 - WA ST DEPT OF COMMERCE,2026-01-06,True,2026-01-06,1,1,0.0
18914,96903,2026-01-26,AR-Research,41341 - KEMPER DEVELOPMENT,2026-01-26,False,1899-12-31,1,0,NaN
18915,97011,2026-02-17,AR-Research,41671 - Walgreen's # 03662,2026-02-17,False,1899-12-31,1,0,NaN


In [28]:
# Create Days to Pay bins

d = df_ClosedInvoice["DaysToPay_FromDue"]

df_ClosedInvoice["DaysToPay_Bin"] = np.select(
    [
        d <= 0,
        (d > 0) & (d <= 30),
        (d > 30) & (d <= 60),
        (d > 60) & (d <= 90),
        d > 90
    ],
    [
        "On time (≤0)",
        "1-30 Days",
        "31-60 Days",
        "61-90 Days",
        "91+ Days"
    ],
    default=pd.NA
)
df_ClosedInvoice

,Document Number,InvoiceDate,BusinessUnitDesc,CustomerNumber,DueDate,PaidInFull,PaidDate,LineCount,ClosedLineCount,DaysToPay_FromDue,DaysToPay_Bin
0,1923,2019-09-11,IT,312721 - CITY OF NEWCASTLE,2019-09-11,True,2019-09-23,1,1,12.0,1-30 Days
1,1926,2019-10-07,Fire,249709 - WA ST Emergency Management Division,2019-10-07,True,2019-10-23,1,1,16.0,1-30 Days
2,1927,2019-10-07,Fire,249709 - WA ST Emergency Management Division,2019-10-07,True,2019-10-09,1,1,2.0,1-30 Days
3,1938,2019-11-14,Parks,632354 - Kelly Edwards / LocalHOOPS Training Acad,2019-11-14,True,2020-02-06,1,1,84.0,61-90 Days
4,1941,2019-12-31,Utilities,319933 - Republic Services,2019-12-31,True,2020-02-05,1,1,36.0,31-60 Days
...,...,...,...,...,...,...,...,...,...,...,...
18912,96855,2026-01-20,AR-Research,846036 - The Compliance Engine (TCE),2026-01-20,False,1899-12-31,1,0,NaN,<NA>
18913,96874,2026-01-06,AR-Research,338190 - WA ST DEPT OF COMMERCE,2026-01-06,True,2026-01-06,1,1,0.0,On time (≤0)
18914,96903,2026-01-26,AR-Research,41341 - KEMPER DEVELOPMENT,2026-01-26,False,1899-12-31,1,0,NaN,<NA>
18915,97011,2026-02-17,AR-Research,41671 - Walgreen's # 03662,2026-02-17,False,1899-12-31,1,0,NaN,<NA>


In [29]:
df_ClosedInvoice["InvoiceDate"] = pd.to_datetime(
    df_ClosedInvoice["InvoiceDate"],
    errors="coerce"
)




In [30]:

min_date = df_ClosedInvoice.loc[
    df_ClosedInvoice["PaidInFull"] == True,
    "InvoiceDate"
].min()



min_date

Timestamp('2002-07-19 00:00:00')

In [31]:
df_ClosedInvoice.loc[
    df_ClosedInvoice["PaidInFull"] == "True",
    "InvoiceDate"
].isna().sum()

np.int64(0)

In [32]:
df_transaction[df_transaction["Document Number"]== "39026"]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Gross Amount,Business Unit,Customer Number,Date Closed,Due Date,Date Last Updated,Remark,Recording Comment,Invoice Point of Contact,Batch Number,Batch Type,Discount Due Date,Date of Last Reminder,Invoice Print Date,Notification Payment Date,Historical Date,Void Date,GL Date,Batch Date,Service Tax Date,Account ID,Delinquency Notice Sent,Number of Reminders Sent,Void Flag,Invoice Point of Contact,Status,BusinessUnitDesc,Days Past Due,Total Open Amount per Document,Aging Bucket,IsClosedLine
4854,39026,0.0,RR,01280,2021-03-01,1666.0,1280,143807 - NORCOM,2021-03-15,2021-04-02,2021-03-16,Monthly Parking Spaces,,Katherine Goetz @ 425-452-7523 ...,419668,IB,121092.0,0.0,123011.0,0.0,0.0,0.0,121060.0,121062.0,121060.0,00006999,Y,0.0,,Katherine Goetz @ 425-452-7523 ...,Closed,NaN,1797,0.0,121+ Days,True
4855,39026,0.0,RR,01280,2021-03-03,166.6,1280,143807 - NORCOM,2021-03-15,2021-04-02,2021-03-16,WA State Sales Tax,,...,419668,IB,121092.0,0.0,123011.0,0.0,0.0,0.0,121060.0,121062.0,121060.0,00006999,Y,0.0,,...,Closed,NaN,1797,0.0,121+ Days,True


### Verification

In [33]:
# # Export to excel for verification
# df_transaction.to_excel("AP report_verification.xlsx", index=False)   


In [34]:
df_transaction["Customer"].isna().any()


KeyError: 'Customer'

In [ ]:
df_transaction["BusinessUnitDesc"].value_counts()

In [ ]:
df_transaction[df_transaction["Document Number"]== 49541]

,Document Number,Combined Open Amount On As Of Date,Document Type,Document Company,Invoice Date,Business Unit,RPAN8,Customer,RPJCL,Date Closed,RPDDJ,Due Date,RPSFX,RPDGJ,RPICUT,RPICU,RPDICJ,RPFY,RPCTRY,RPPN,RPCO,RPGLC,RPAID,RPPA8,RPAN8J,RPPYR,RPPOST,RPISTR,RPBALJ,RPPST,RPADSC,RPADSA,RPATXA,RPATXN,RPSTAM,RPBCRC,RPCRRM,RPCRCD,RPCRR,RPDMCD,RPACR,RPFAP,RPCDS,RPCDSA,RPCTXA,RPCTXN,RPCTAM,RPTXA1,RPEXR1,RPDSVJ,RPGLBA,RPAM,RPAID2,RPAM2,RPOBJ,RPSUB,RPSBLT,RPSBL,RPPTC,RPDDNJ,RPRDDJ,RPRDSJ,RPLFCJ,RPSMTJ,RPNBRR,RPRDRL,RPRMDS,RPCOLL,RPCORC,RPAFC,RPDNLT,RPRSCO,RPODOC,RPODCT,RPOKCO,RPOSFX,RPVINV,RPPO,RPPDCT,RPPKCO,RPDCTO,RPLNID,RPSDOC,RPSDCT,RPSKCO,RPSFXO,RPVLDT,RPCMC1,RPVR01,RPUNIT,RPMCU2,RPRMK,RPALPH,RPRF,RPDRF,RPCTL,RPFNLP,RPITM,RPU,RPUM,RPALT6,RPRYIN,RPVDGJ,RPVOD,RPRP1,RPRP2,RPRP3,RPAR01,RPAR02,RPAR03,RPAR04,RPAR05,RPAR06,RPAR07,RPAR08,RPAR09,RPAR10,RPISTC,RPPYID,RPURC1,RPURDT,RPURAT,RPURAB,RPURRF,RPRNID,RPPPDI,RPTORG,RPUSER,RPJCL,RPPID,RPUPMJ,RPUPMT,RPDDEX,RPJOBN,RPHCRR,RPHDGJ,RPSHPN,RPDTXS,RPOMOD,RPCLMG,RPCMGR,RPATAD,RPCTAD,RPNRTA,RPFNRT,RPPRGF,RPGFL1,RPGFL2,RPDOCO,RPKCOO,RPSOTF,RPDTPB,RPERDJ,RPPWPG,RPNETTCID,RPNETDOC,RPNETRC5,RPNETST,RPAJCL,RPRMR1,BusinessUnitDesc,Days Past Due,Total Payment per Document,Aging Bucket


In [ ]:
# Verification:

doc_list = [
2251, 53555, 53582, 53583, 53584, 53591, 53592, 53593, 53594, 53609,
53612, 53618, 53636, 53639, 53647, 53648, 53672, 53690, 53696, 53742,
53744, 53745, 53754, 53764, 53765, 53766, 53769, 53771, 53777, 53779,
53780, 53792, 53795, 53801, 53817, 53823, 53824, 53833, 53835, 53846,
53850, 53851, 53855, 53856, 53861, 53862, 53864, 53865, 53872, 53874,
53880, 53881, 53884, 53885, 53888, 53890, 53898, 53899, 53904, 53909,
53911, 53919, 53922, 53924, 53928, 53931, 53933, 53935, 53943, 53952,
53959, 53960, 53961, 53962, 53964, 53965, 53968, 53970, 53971, 53974,
53975, 53978, 53980, 53984, 53989, 53990, 53993, 53994, 53998, 54000,
54002, 54006, 54007, 54008, 54009, 54010, 54011, 54012, 54013, 54014,
54017, 54019, 54020, 54021, 54022, 54025, 54027, 54028, 54029, 54031,
54032, 54034, 54036, 54039, 54040, 54042, 54047, 54048, 54050, 54057,
54060, 54062, 54064, 54065, 54066, 54067, 54068, 54069, 54072, 54075,
54076, 54081, 54082, 54085, 54086, 54089, 54090, 54093, 54096, 54097,
54101, 54102, 54104, 54105, 54106, 54109, 54110, 54112, 54120, 54121,
54123, 54126, 54127, 54129, 54130, 54136, 54137, 54149, 54150, 54151,
54153, 54156, 54157, 54161, 54162, 54163, 54166, 54167, 54168, 54169,
54170, 54171, 54172, 54173, 54174, 54175, 54176, 54177, 54178, 54179,
54180, 54181, 54182, 54183, 54184, 54185, 54186, 54187, 54188, 54189,
54190, 54192, 54193, 54194, 54195, 54196, 54197, 54202, 54203, 54204,
54207, 54209, 54264, 54265, 54267, 54268, 54270, 54273, 54275, 54277,
54279, 54280, 54282, 54283, 54286, 54287, 54289, 54290, 54294, 54295,
54296, 54297, 54306, 54308, 54309, 54311, 54344, 54346, 54348, 54349,
54350, 54361, 54368, 54379, 54381, 54382, 54384, 54386, 54387, 54389,
54390, 54391, 54396, 54427, 54428, 54429, 54430, 54431, 54433, 54434,
54435, 54436, 54437, 54439, 54440, 54442, 54443, 54444, 54445, 54446,
54447, 54448, 54449, 54450, 54452, 54453, 54455, 54456, 54458, 54460,
54461, 54462, 54463, 54465, 54466, 54467, 54468, 54470, 54471, 54472,
54473, 54474, 54475, 54477, 54479, 54480, 54482, 54483, 54484, 54485,
54486, 54487, 54488, 54490, 54492, 54493, 54494, 54495, 54496, 54497,
54498, 54499, 54500, 54501, 54502, 54503, 54504, 54506, 54507, 54508,
54509, 54513, 54514, 54515, 54516, 54517, 54518, 54519, 54520, 54521,
54522, 54526, 54528, 54531, 54532, 54533, 54535, 54536, 54537, 54538,
54539, 54540, 54544, 54545, 54546, 54547, 54548, 54549, 54550, 54551,
54552, 54553, 54554, 54555, 54556, 54557, 54558, 54559, 54560, 54561,
54562, 54563, 54564, 54565, 54566, 54567, 54568, 54569, 54570, 54571,
54572, 54573, 54574, 54575, 54576, 54577, 54578, 54579, 54580, 54581,
54582, 54583, 54584, 54585, 54586, 54587, 54588, 54589, 54590, 54591,
54592, 54593, 54594, 54595, 54596, 54597, 54598, 54599, 54600, 54601,
54602, 54603, 54604, 54605, 54606, 54607, 54608, 54609, 54610, 54611,
54612, 54613, 54616, 54617, 54618, 54619, 54620, 54621, 54622, 54623,
54624, 54625, 54626, 54627, 54628, 54629, 54630, 54631, 54632, 54633,
54634, 54635, 54636, 54637, 54638, 54639, 54640, 54641, 54642, 54643,
54644, 54645, 54646, 54647, 54648, 54649, 54650, 54651, 54652, 54653,
54654, 54655, 54656, 54657, 54658, 54659, 54660, 54661, 54662, 54663,
54664, 54665, 54666, 54667, 54668, 54669, 54670, 54671, 54672, 54673,
54674, 54675, 54676, 54677, 54678, 54679, 54680, 54681, 54682, 54683,
54684, 54685, 54686, 54687, 54688, 54689, 54690, 54691, 54692, 54693,
54694, 54695, 54696, 54697, 54698, 54699, 54700, 54701, 54702,
96472, 96473, 96483, 96486, 96488, 96604, 96622, 96655, 96659, 96678,
96772, 96834, 96835, 96836, 96837, 96855, 96903
]
filtered = df_transaction[df_transaction["Document Number"].isin(doc_list)]


In [ ]:
filtered["Invoice Date"].value_counts()

Invoice Date
2026-02-27    105
2026-02-03     50
2025-12-16     40
2026-01-06     33
2026-01-16     23
2025-12-12     21
2026-01-27     19
2026-01-21     15
2025-12-17     14
2025-12-30     12
2025-11-13     12
2025-12-15     12
2025-12-23     10
2025-11-25      9
2025-11-24      8
2026-01-26      7
2026-01-23      6
2026-01-09      6
2026-01-12      6
2025-12-19      6
2025-12-22      6
2025-11-14      6
2025-11-26      6
2025-12-11      5
2026-01-02      5
2026-01-20      5
2025-10-01      4
2026-01-15      4
2025-12-18      4
2025-09-30      3
2025-10-20      3
2025-12-26      3
2025-12-03      3
2025-12-09      2
2026-01-29      2
2025-10-27      2
2025-11-07      2
2025-10-23      2
2025-10-22      2
2025-11-17      2
2026-01-01      2
2025-12-01      2
2025-10-08      1
2025-12-04      1
2025-12-10      1
2025-12-05      1
2025-11-05      1
2025-10-10      1
2025-10-09      1
2025-10-16      1
2025-10-15      1
2025-09-29      1
2026-01-05      1
2025-12-25      1
2026-01-08     

In [ ]:
(filtered["Invoice Date"] >= pd.Timestamp("2025-09-29")).all()


np.True_

In [ ]:
df_transaction["Document Number"] = df_transaction["Document Number"].astype(int)


In [ ]:
df_transaction.info()

In [ ]:
df_transaction.describe()

In [ ]:
df_transaction[df_transaction['Document Number'] == 53833]

In [ ]:
df_invoice[df_invoice["Document Number"] =="48823"]

,Document Number,InvoiceDate,BusinessUnitDesc,CustomerNumber,DueDate,PaidInFull,PaidDate,LineCount,ClosedLineCount,DaysToPay_FromDue
12498,48823,2024-03-28,Transportation,960517 - Verizon Wireless,2024-04-27,False,2024-07-03,2,1,NaN
